In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q decord

import os
import glob
import random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from transformers import TimesformerModel, TimesformerConfig
from decord import VideoReader, cpu
from tqdm import tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 76.9 MB/s eta 0:00:00


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
class FightDataset(Dataset):
    def __init__(self, root_dir, num_frames=8, mode="train"):
        self.samples = []
        self.num_frames = num_frames
        self.mode = mode

        self.label_map = {
            "NonViolence": 0,
            "Violence": 1
        }

        for cls, label in self.label_map.items():
            for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv", "*.mpeg"]:
                for path in glob.glob(os.path.join(root_dir, cls, ext)):
                    self.samples.append((path, label))

        random.shuffle(self.samples)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.samples)

    def _sample_indices(self, total_frames):
        if total_frames <= self.num_frames:
            return np.linspace(0, total_frames - 1, self.num_frames).astype(int)

        if self.mode == "train":
            start = random.randint(0, total_frames - self.num_frames)
        else:
            start = (total_frames - self.num_frames) // 2

        return np.arange(start, start + self.num_frames)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]

        vr = VideoReader(video_path, ctx=cpu(0))
        indices = self._sample_indices(len(vr))
        frames = vr.get_batch(indices).asnumpy()

        frames = [self.transform(f) for f in frames]
        video = torch.stack(frames).permute(1, 0, 2, 3)  # [C, T, H, W]

        return video, torch.tensor(label)


In [ ]:
TRAIN_ROOT = "/content/drive/MyDrive/NewTrainingDataset"

train_dataset = FightDataset(
    TRAIN_ROOT,
    mode="train"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Total training samples:", len(train_dataset))


Total training samples: 2075


In [ ]:
class FightTimeSformer(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)


config = TimesformerConfig.from_pretrained(
    "facebook/timesformer-base-finetuned-k400"
)
config.num_frames = 8
config.image_size = 224

backbone = TimesformerModel.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    config=config
)

model = FightTimeSformer(backbone)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/securevision_fight_timesformer.pth",
               map_location=device)
)

model = model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/486M [00:00<?, ?B/s]

In [ ]:
# Freeze entire backbone
for p in model.backbone.parameters():
    p.requires_grad = False

# Unfreeze LAST encoder block
for p in model.backbone.encoder.layer[-1].parameters():
    p.requires_grad = True

# Classifier always trainable
for p in model.classifier.parameters():
    p.requires_grad = True


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,          # LOW LR (important)
    weight_decay=1e-4
)


In [ ]:
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        videos = videos.to(device)
        labels = labels.to(device)

        videos = videos.permute(0, 2, 1, 3, 4)  # [B, T, C, H, W]

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f}")


Epoch 1/2: 100%|██████████| 519/519 [14:01<00:00,  1.62s/it]


Epoch 1 | Train Loss: 0.0907


Epoch 2/2: 100%|██████████| 519/519 [07:32<00:00,  1.15it/s]

Epoch 2 | Train Loss: 0.0446


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/securevision_fight_timesformer_hardneg.pth"
)

print("✅ Hard-negative fine-tuned model saved")


✅ Hard-negative fine-tuned model saved
